# XGBOOST

In [ ]:

import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, classification_report

# 1. Create the dataset

# Features: [Hours studied, Hours slept]
X = np.array([
    [1, 5],
    [2, 6],
    [2, 4],
    [3, 7],
    [4, 6],
    [5, 8],
    [6, 7],
    [7, 8],
    [8, 6],
    [9, 8],
    [3, 5],
    [6, 6],
    [7, 5],
    [4, 8],
    [8, 9],
    [5, 5],
    [9, 6],
    [2, 7],
    [6, 9],
    [7, 7]
], dtype=float)

# 0 = Fail, 1 = Pass
y = np.array([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    0, 1, 1, 0, 1,
    0, 1, 0, 1, 1
])

# 2. Split training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# 3. Create the XGBoost model

model = XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=2,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

# 4. Train

model.fit(X_train, y_train)

# 5. Evaluate

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Log Loss:", round(log_loss(y_test, y_prob), 3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 6. Predict new students

X_new = np.array([
    [2, 5],
    [5, 7],
    [8, 7]
], dtype=float)

probabilities = model.predict_proba(X_new)[:, 1]
predictions = model.predict(X_new)

print("\nNew Predictions:")

for features, probability, prediction in zip(
    X_new, probabilities, predictions
):
    result = "Pass" if prediction == 1 else "Fail"

    print(
        f"Features: {features}, "
        f"Pass probability: {probability:.3f}, "
        f"Prediction: {result}"
    )


Accuracy: 1.0
Log Loss: 0.199

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5


New Predictions:
Features: [2. 5.], Pass probability: 0.197, Prediction: Fail
Features: [5. 7.], Pass probability: 0.500, Prediction: Fail
Features: [8. 7.], Pass probability: 0.831, Prediction: Pass


# XGBoost From Scratch

In [ ]:

import numpy as np


# 1. XGBOOST TREE
class XGBoostTree:
    def __init__(self, max_depth=2, min_samples_split=2,
                 reg_lambda=1.0, gamma=0.0):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.reg_lambda = reg_lambda
        self.gamma = gamma

        self.tree = None

    def _sigmoid(self, scores):
        scores = np.clip(scores, -500, 500)
        return 1 / (1 + np.exp(-scores))

    def _leaf_weight(self, gradients, hessians):
        G = np.sum(gradients)
        H = np.sum(hessians)

        return -G / (H + self.reg_lambda)

    def _gain(self, gradients, hessians):
        G = np.sum(gradients)
        H = np.sum(hessians)

        return (G ** 2) / (H + self.reg_lambda)

    def _build_tree(self, X, gradients, hessians, depth):
        # Default prediction for this node
        leaf_value = self._leaf_weight(gradients, hessians)

        node = {
            "leaf_value": leaf_value,
            "feature": None,
            "threshold": None,
            "left": None,
            "right": None
        }

        # Stop conditions
        if depth >= self.max_depth:
            return node

        if len(X) < self.min_samples_split:
            return node

        parent_gain = self._gain(gradients, hessians)
        best_gain = 0
        best_split = None

        n_samples, n_features = X.shape

        # Search every feature and threshold
        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask

                if not np.any(left_mask) or not np.any(right_mask):
                    continue

                G_left = np.sum(gradients[left_mask])
                H_left = np.sum(hessians[left_mask])

                G_right = np.sum(gradients[right_mask])
                H_right = np.sum(hessians[right_mask])

                left_gain = (G_left ** 2) / (
                    H_left + self.reg_lambda
                )

                right_gain = (G_right ** 2) / (
                    H_right + self.reg_lambda
                )

                gain = 0.5 * (
                    left_gain
                    + right_gain
                    - parent_gain
                ) - self.gamma

                if gain > best_gain:
                    best_gain = gain
                    best_split = (feature, threshold, left_mask)

        # Keep this node as a leaf if no useful split exists
        if best_split is None:
            return node

        feature, threshold, left_mask = best_split
        right_mask = ~left_mask

        node["feature"] = feature
        node["threshold"] = threshold

        node["left"] = self._build_tree(
            X[left_mask],
            gradients[left_mask],
            hessians[left_mask],
            depth + 1
        )

        node["right"] = self._build_tree(
            X[right_mask],
            gradients[right_mask],
            hessians[right_mask],
            depth + 1
        )

        return node

    def fit(self, X, gradients, hessians):
        self.tree = self._build_tree(
            X, gradients, hessians, depth=0
        )

    def _predict_one(self, row, node):
        if node["feature"] is None:
            return node["leaf_value"]

        if row[node["feature"]] <= node["threshold"]:
            return self._predict_one(row, node["left"])

        return self._predict_one(row, node["right"])

    def predict(self, X):
        return np.array([
            self._predict_one(row, self.tree)
            for row in X
        ])


# 2. XGBOOST CLASSIFIER

class XGBoostClassifier:
    def __init__(self, n_estimators=10, learning_rate=0.3,
                 max_depth=2, reg_lambda=1.0, gamma=0.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.reg_lambda = reg_lambda
        self.gamma = gamma

        self.trees = []
        self.base_score = 0.0

    def sigmoid(self, scores):
        scores = np.clip(scores, -500, 500)
        return 1 / (1 + np.exp(-scores))

    def fit(self, X, y):
        # Start from the log odds of the positive class
        p = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        self.base_score = np.log(p / (1 - p))

        scores = np.full(len(y), self.base_score)
        self.trees = []
    
        for i in range(self.n_estimators):
            probabilities = self.sigmoid(scores)

            # First derivative of binary logistic loss
            gradients = probabilities - y

            # Second derivative of binary logistic loss
            hessians = probabilities * (1 - probabilities)

            tree = XGBoostTree(
                max_depth=self.max_depth,
                reg_lambda=self.reg_lambda,
                gamma=self.gamma
            )

            tree.fit(X, gradients, hessians)

            # Update raw scores
            scores += self.learning_rate * tree.predict(X)

            self.trees.append(tree)

            # Binary cross-entropy loss
            p_pred = np.clip(
                self.sigmoid(scores), 1e-10, 1 - 1e-10
            )

            loss = -np.mean(
                y * np.log(p_pred)
                + (1 - y) * np.log(1 - p_pred)
            )

            print(
                f"Iteration {i + 1:02d} | "
                f"Log Loss: {loss:.4f}"
            )

    def predict_proba(self, X):
        scores = np.full(len(X), self.base_score)

        for tree in self.trees:
            scores += self.learning_rate * tree.predict(X)

        probabilities = self.sigmoid(scores)

        return np.column_stack([
            1 - probabilities,
            probabilities
        ])

    def predict(self, X):
        return (
            self.predict_proba(X)[:, 1] >= 0.5
        ).astype(int)


# 3. DATASET

# Features: [Hours studied, Hours slept]
X = np.array([
    [1, 5],
    [2, 6],
    [2, 4],
    [3, 7],
    [4, 6],
    [5, 8],
    [6, 7],
    [7, 8],
    [8, 6],
    [9, 8],
    [3, 5],
    [6, 6],
    [7, 5],
    [4, 8],
    [8, 9],
    [5, 5],
    [9, 6],
    [2, 7],
    [6, 9],
    [7, 7]
], dtype=float)

# 0 = Fail, 1 = Pass
y = np.array([
    0, 0, 0, 0, 0,
    1, 1, 1, 1, 1,
    0, 1, 1, 0, 1,
    0, 1, 0, 1, 1
])

# 4. TRAIN

model = XGBoostClassifier(
    n_estimators=10,
    learning_rate=0.3,
    max_depth=2,
    reg_lambda=1.0,
    gamma=0.0
)

model.fit(X, y)


# 5. PREDICT

X_test = np.array([
    [2, 5],
    [5, 7],
    [8, 7]
], dtype=float)

probabilities = model.predict_proba(X_test)
predictions = model.predict(X_test)

print("\nFinal Predictions:")

for features, probability, prediction in zip(
    X_test, probabilities[:, 1], predictions
):
    result = "Pass" if prediction == 1 else "Fail"

    print(
        f"Features: {features}, "
        f"Pass Probability: {probability:.3f}, "
        f"Prediction: {result}"
    )


Iteration 01 | Log Loss: 0.5221
Iteration 02 | Log Loss: 0.4103
Iteration 03 | Log Loss: 0.3293
Iteration 04 | Log Loss: 0.2716
Iteration 05 | Log Loss: 0.2268
Iteration 06 | Log Loss: 0.1935
Iteration 07 | Log Loss: 0.1664
Iteration 08 | Log Loss: 0.1458
Iteration 09 | Log Loss: 0.1283
Iteration 10 | Log Loss: 0.1148

Final Predictions:
Features: [2. 5.], Pass Probability: 0.086, Prediction: Fail
Features: [5. 7.], Pass Probability: 0.469, Prediction: Fail
Features: [8. 7.], Pass Probability: 0.931, Prediction: Pass
